# LangGraph POC — Live LLM Walkthrough

This notebook runs the **real, live** version of the repository's LangGraph POC. It mirrors the `--live` wiring in `agentrun_langgraph_poc.py`, but exposes the setup and each outer graph-node update so you can learn from the execution.

## What this notebook does

1. Loads your OpenAI API key from the environment or the repository's `.env` file.
2. Creates the project's real `OpenAILLMClient` for the planner.
3. Creates a real `ChatOpenAI` model for the LangChain `create_agent` Investigator and a separate tool-free Analyst.
4. Binds the project's existing read-only repository tools to a repository root you choose.
5. Builds the `StateGraph`, then streams its outer-node updates.
6. Displays P1-P3 reports, the P4 synthesis, deterministic validation, and middleware instrumentation.

## Important boundaries

- **This makes real API calls and can incur usage charges.** The planner calls the model, and the Investigator may make multiple calls while using tools.
- This M5.2 notebook executes **P1, P2, and P3 sequentially** as separate Investigator invocations, then runs P4 as a tool-free Analyst over the three reports. A deterministic validator checks scope, source-report IDs, and evidence provenance.
- The tools are the repository's existing tools; this notebook does not invent replacement tool implementations.
- The graph is read-only with respect to repository content, but model/API calls and local logging may still occur.
- Run this notebook from the repository's root, or set `REPOSITORY_ROOT` in the configuration cell below.
- Do not put API keys directly into notebook cells or commit `.env`.

**This notebook is intended for interactive learning, not as proof that the full test suite passes.** The notebook file can be syntax-checked here, but the live run must be executed in your environment with the required dependencies and API access.

## 1. Install the POC dependencies

Run this once in a terminal from the repository root (not necessarily in this notebook):

```bash
pip install -e ".[poc]"
```

The repository's `poc` extra installs LangGraph, LangChain, and `langchain-openai`. The project also needs its `openai` dependency for the planner client. If it is not installed by your environment already, install the project's `llm` extra too:

```bash
pip install -e ".[poc,llm]"
```

Then launch Jupyter from the repository root:

```bash
jupyter lab
```

If you installed dependencies while Jupyter was already running, restart the notebook kernel before continuing.

## 2. Configure the target repository and question

The notebook's Python package and the repository being investigated are
different things.

- `PROJECT_ROOT` is this repository-intelligence project. It provides the
  POC implementation and Python modules.
- `REPOSITORY_ROOT` is the pinned sample-Agentic-Ai-Data-Operations
  snapshot being investigated.
- `ARTIFACTS_ROOT` contains the existing index and architecture artifacts
  for that pinned snapshot.

The live investigator's tools must be bound to `REPOSITORY_ROOT`, not
`PROJECT_ROOT`.

In [1]:
from pathlib import Path
import os
import sys
import json
import pprint
import traceback

# The project that implements the repository-intelligence tools.
PROJECT_ROOT = Path.cwd().resolve()

# Make the project's Python package available to this notebook.
project_src = str(PROJECT_ROOT / "src")
if project_src not in sys.path:
    sys.path.insert(0, project_src)

from repo_intelligence.repository_query import (
    default_subject_repository,
    default_artifacts_root,
)
from repo_intelligence.retrieval.identity import resolve_index_identity

# The pinned repository we want the agent to investigate.
REPOSITORY_ROOT = default_subject_repository().resolve()
ARTIFACTS_ROOT = default_artifacts_root().resolve()

# Resolve provenance from the existing snapshot record.
identity = resolve_index_identity(
    REPOSITORY_ROOT,
    output_root=ARTIFACTS_ROOT,
)

QUESTION = "How do the agents, tools, and MCP servers interact?"

REPOSITORY_SCOPE = identity.repository_name
COMMIT_SCOPE = identity.commit_sha

print("Project root:", PROJECT_ROOT)
print("Target repository:", REPOSITORY_ROOT)
print("Artifacts root:", ARTIFACTS_ROOT)
print("Repository:", identity.repository_name)
print("Commit:", identity.commit_sha)
print("Snapshot:", identity.snapshot_id)
print("Contains target README:", (REPOSITORY_ROOT / "README.md").is_file())
print("Python:", sys.version.split()[0])

Project root: C:\self-study\ai\build\repository-intelligence
Target repository: C:\self-study\ai\build\repository-intelligence\artifacts\main_2026-09-22_ef11dac\repository\sample-Agentic-Ai-Data-Operations
Artifacts root: C:\self-study\ai\build\repository-intelligence\artifacts
Repository: sample-Agentic-Ai-Data-Operations
Commit: ef11dac8618b6344d6a5e4add6ef51cd2811d7e0
Snapshot: main_2026-09-22_ef11dac
Contains target README: True
Python: 3.11.5


### Check before proceeding

The two repository checks above should both print `True`. If either is `False`, update `REPOSITORY_ROOT` and rerun the configuration cell before continuing. Correct repository scope matters: the live tools should inspect the repository you intend to investigate.

## 3. Load credentials safely

The project provides `load_env()` and `get_openai_api_key()`. This cell loads `.env` from the configured repository root if present, without overriding environment variables that are already set. It never prints the key.

In [2]:

# Validate the analyzer project root before importing project-specific configuration.
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError(
        f"{PROJECT_ROOT} does not look like the analyzer project root: "
        "pyproject.toml is missing. Check PROJECT_ROOT in the configuration cell."
    )

if not (PROJECT_ROOT / "src" / "repo_intelligence").is_dir():
    raise FileNotFoundError(
        f"{PROJECT_ROOT} does not contain src/repo_intelligence. "
        "Check PROJECT_ROOT in the configuration cell."
    )

# Make the analyzer's source tree importable.
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from repo_intelligence.llm.config import load_env, get_openai_api_key

# Load configuration from the analyzer project, not the target repository.
loaded_from_dotenv = load_env(PROJECT_ROOT / ".env")
api_key = get_openai_api_key()

print("Loaded new environment values from .env:", sorted(loaded_from_dotenv.keys()))
print("OPENAI_API_KEY configured:", bool(api_key))

if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY is not configured. Set it in your shell environment "
        "or create "
        f"{PROJECT_ROOT / '.env'} with OPENAI_API_KEY=... . "
        "Do not paste the key into this notebook."
    )

# Separately validate the target repository.
if not (REPOSITORY_ROOT / "README.md").is_file():
    raise FileNotFoundError(
        f"Target repository README.md is missing: {REPOSITORY_ROOT}"
    )


Loaded new environment values from .env: []
OPENAI_API_KEY configured: True


## 4. Construct the live planner, model, tools, and Investigator

This is the notebook equivalent of the CLI script's `_build_live()` function. It uses the same model constant, project planner client, `live_bindings()`, and `build_create_agent_investigator()` factory.

The middleware is installed by `build_create_agent_investigator`; it includes request-budget enforcement, execution limits, and bounded tool observations. We inspect its records later rather than replacing or bypassing it.

> **M5.2 scope:** P1-P3 run sequentially with separate reports; P4 synthesizes them with no repository tools, then deterministic validation checks provenance and structure.


In [3]:
from langchain_openai import ChatOpenAI

from repo_intelligence.llm.openai_client import DEFAULT_MODEL, OpenAILLMClient
from repo_intelligence.repository_query import live_bindings
from repo_intelligence.langgraph_poc import (
    build_create_agent_analyst,
    build_create_agent_investigator,
    build_poc_graph,
    GRAPH_RECURSION_LIMIT,
)

print("Configured model:", DEFAULT_MODEL)

# Planner: project-owned provider-neutral LLMClient implementation.
planner = OpenAILLMClient(model=DEFAULT_MODEL, max_retries=0)

# Investigator model is constructed below with parallel tool calls disabled.
# Existing project tools, bound to the selected repository root.
bindings = live_bindings(
    REPOSITORY_ROOT,
    output_root=ARTIFACTS_ROOT,
)

# Use a task-general prompt. The previous notebook hard-coded a README-only
# instruction here, so changing QUESTION caused the investigator to keep
# re-reading README ranges instead of investigating the new question.
#
# Disable parallel tool calls so a single model response cannot consume most
# of the eight-call safety budget by launching many read_file calls at once.
investigator_model = ChatOpenAI(
    model=DEFAULT_MODEL,
    api_key=api_key,
    model_kwargs={"parallel_tool_calls": False},
)

INVESTIGATION_PROMPT = """
ROLE
You are a read-only repository investigator. Return one structured
InvestigationReport matching the supplied schema, not an ArchitectureReport.

TASK AND SCOPE
The TASK contains the full question, repository_scope, commit_scope, and
step_goal. Focus this run on the assigned step_goal while using the question
as context. Investigate only the pinned repository and commit.

EVIDENCE
- Use search_repository for discovery, then read the exact ranges you cite
  with read_file. Search results alone are not evidence.
- Choose one high-value search query using the key concepts in the question
  and step_goal. Do not repeatedly search synonyms.
- Read no more than four relevant source ranges. Do not request multiple
  read_file calls in parallel; inspect one result, reassess, then continue
  only if a material evidence gap remains.
- Do not re-read overlapping ranges unless the earlier output was truncated
  or a specific unresolved fact requires it.
- Do not inspect unrelated files or call architecture/graph tools unless
  the assigned step explicitly requires them.
- Stop as soon as evidence is sufficient. If something is not established,
  report uncertainty rather than continuing exploratory calls.

REPORT CONTRACT
- Echo question, repository_scope, and commit_scope exactly from the TASK.
- Provide a non-empty investigation_id.
- Each finding must be evidence-backed and include evidence_refs and support.
- Every cited path/range must have been read during this run.
- evidence_refs must include the references used by findings.
- investigated_sources lists the paths actually read.
- Use status "complete" only if the assigned step is answered; otherwise
  use "incomplete". Include open_questions and uncertainties when needed.
"""

investigator = build_create_agent_investigator(
    investigator_model,
    bindings,
    prompt=INVESTIGATION_PROMPT,
    tool_names=["search_repository", "read_file"],
)

# P4 Analyst: separate create_agent, no repository tools.
analyst_model = ChatOpenAI(
    model=DEFAULT_MODEL,
    api_key=api_key,
    model_kwargs={"parallel_tool_calls": False},
)
analyst = build_create_agent_analyst(analyst_model)

print("Bound tool names:", sorted(bindings.keys()))
print("Investigator tool names:", investigator.tool_names)
print("Outer graph recursion limit:", GRAPH_RECURSION_LIMIT)
print("Investigator middleware handles available:",
      {"budget": investigator.budget is not None,
       "execution": investigator.execution is not None,
       "observation": investigator.observation is not None})


Configured model: gpt-4o-mini
Bound tool names: ['get_architecture_context', 'query_engineering_graph', 'read_file', 'search_repository']
Investigator tool names: ['search_repository', 'read_file']
Outer graph recursion limit: 25
Investigator middleware handles available: {'budget': True, 'execution': True, 'observation': True}


In [4]:
assert REPOSITORY_ROOT.is_dir(), REPOSITORY_ROOT
assert (REPOSITORY_ROOT / "README.md").is_file()
assert identity.repository_name == "sample-Agentic-Ai-Data-Operations"
assert identity.commit_sha == "ef11dac8618b6344d6a5e4add6ef51cd2811d7e0"
assert identity.snapshot_id == "main_2026-09-22_ef11dac"

assert REPOSITORY_SCOPE == identity.repository_name
assert COMMIT_SCOPE == identity.commit_sha

expected_tools = {
    "read_file",
    "search_repository",
    "get_architecture_context",
    "query_engineering_graph",
}
assert expected_tools.issubset(bindings.keys())

print("PASS: target repository exists")
print("PASS: pinned repository identity matches")
print("PASS: scope metadata matches the pinned identity")
print("PASS: all expected tools are bound")
print("Repository:", REPOSITORY_SCOPE)
print("Commit:", COMMIT_SCOPE)
print("Snapshot:", identity.snapshot_id)

PASS: target repository exists
PASS: pinned repository identity matches
PASS: scope metadata matches the pinned identity
PASS: all expected tools are bound
Repository: sample-Agentic-Ai-Data-Operations
Commit: ef11dac8618b6344d6a5e4add6ef51cd2811d7e0
Snapshot: main_2026-09-22_ef11dac


### What each object is responsible for

- **`planner`** — creates the four-step plan through the existing provider-neutral `LLMClient`.
- **`investigator`** — runs P1–P3 with repository tools and the existing safety middleware.
- **`analyst`** — runs P4 using only the three validated investigation reports; it has no repository tools.
- **`build_poc_graph()`** — composes planning, role assignment, investigation, synthesis, and deterministic validation.

If construction fails, inspect the traceback before continuing. Do not substitute an offline model in this live notebook.


## 5. Build the outer graph

The M5.2 graph has explicit nodes and routing:

```text
START → planner → query_router → orchestrator_select
      → investigator → orchestrator_receive
          ↘ repeat P1–P3 until all three reports are complete
      → analyst (P4; no repository tools)
      → deterministic_validation
      → final_response → END
```

P4 is not allowed to start unless all three Investigator reports are complete. Deterministic validation checks repository/commit scope, report IDs, evidence provenance, conclusion-to-finding links, and output structure. It validates provenance and structure, not natural-language entailment.


In [5]:
graph = build_poc_graph(
    planner_client=planner,
    investigator=investigator,
    analyst=analyst,
    multi_step=True,  # M5.2: execute P1-P3, then P4 synthesis + validation
)

print("Compiled graph type:", type(graph).__name__)
print("Question:", QUESTION)


Compiled graph type: CompiledStateGraph
Question: How do the agents, tools, and MCP servers interact?


## 6. Run the live graph and inspect node updates

**This is the cell that starts live execution and can incur API charges.** It invokes the graph once and consumes the stream once, avoiding a second API run just to inspect the final state.

Each emitted item is an outer graph node update. The Investigator's internal model/tool loop is bounded by its own middleware; this outer stream reports the completed Investigator-node update, not every internal token.

In [6]:
from langchain_core.callbacks import BaseCallbackHandler

class InvestigatorTrace(BaseCallbackHandler):
    def on_chat_model_start(self, serialized, messages, **kwargs):
        print("\n[MODEL START]")

    def on_llm_end(self, response, **kwargs):
        print("[MODEL END]")
        for generation in response.generations:
            for item in generation:
                message = getattr(item, "message", None)
                if message is None:
                    continue

                calls = getattr(message, "tool_calls", []) or []
                for call in calls:
                    print(
                        "  TOOL CALL:",
                        call.get("name"),
                        "ARGS:",
                        call.get("args"),
                    )

                content = getattr(message, "content", "")
                if content:
                    print("  RESPONSE:", str(content)[:1500])

    def on_tool_start(self, serialized, input_str, **kwargs):
        print(f"\n[TOOL START] {serialized.get('name', 'unknown')}")
        print("  INPUT:", str(input_str)[:500])

    def on_tool_end(self, output, **kwargs):
        print("[TOOL END]", str(output)[:500])

    def on_tool_error(self, error, **kwargs):
        print("[TOOL ERROR]", repr(error))

    def on_llm_error(self, error, **kwargs):
        print("[MODEL ERROR]", repr(error))

In [7]:
initial_state = {
    "question": QUESTION,
    "repository_scope": REPOSITORY_SCOPE,
    "commit_scope": COMMIT_SCOPE,
}

state_snapshot = dict(initial_state)
node_updates = []
trace = InvestigatorTrace()
try:
    for update in graph.stream(
        initial_state,
        config={"recursion_limit": GRAPH_RECURSION_LIMIT,
                "callbacks": [trace],},
        stream_mode="updates",
    ):
        node_updates.append(update)
        for node_name, node_update in update.items():
            # Merge the outer node's returned fields into a local snapshot
            # so the intermediate state can be inspected without rerunning.
            if isinstance(node_update, dict):
                state_snapshot.update(node_update)

            print("\n" + "=" * 78)
            print("GRAPH NODE UPDATE:", node_name)
            print("=" * 78)
            pprint.pp(node_update, sort_dicts=False)

except Exception as exc:
    print("\nLive graph execution raised an exception:")
    print(f"{type(exc).__name__}: {exc}")
    traceback.print_exc()
    raise

print("\nNumber of outer node updates:", len(node_updates))



GRAPH NODE UPDATE: planner
{'plan': InvestigationPlan(objective='Understand the interactions between agents, tools, and MCP servers.', steps=[PlanStep(id='P1', goal='Identify the role and functionality of agents in the system.', status='pending', result=None), PlanStep(id='P2', goal='Explore the types of tools utilized by the agents and their purpose.', status='pending', result=None), PlanStep(id='P3', goal='Examine how MCP servers facilitate communication and data exchange between agents and tools.', status='pending', result=None), PlanStep(id='P4', goal='Synthesize the findings to illustrate the overall interaction dynamics among agents, tools, and MCP servers.', status='pending', result=None)], status='pending'),
 'routing': 'continue'}

GRAPH NODE UPDATE: query_router
{'role_assignments': [RoleAssignment(step_id='P1',
                                     role='INVESTIGATOR',
                                     reason='application-assigned role for '
                              

c:\self-study\ai\build\repository-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2391.53it/s]


[TOOL END] content='{"data": {"evidence_note": "results are discoveries, not evidence: each result names a candidate path and line range \\u2014 call read_file on the exact range you intend to cite before putting it in evidence_refs", "note": null, "omitted_results": 0, "query": "agents functionality", "results": [{"chunk_id": "6ed66f60e9b5ef31", "commit_sha": "ef11dac8618b6344d6a5e4add6ef51cd2811d7e0", "excerpt": "# SKILLS.md \\u2014 Agentic Data Onboarding System\\n\\n> Claude Code skill definitions for 

[MODEL START]
[MODEL END]
  TOOL CALL: read_file ARGS: {'path': 'runbooks/data-onboarding-agent/README.md', 'line_start': 1, 'line_end': 40}

[TOOL START] read_file
  INPUT: {'path': 'runbooks/data-onboarding-agent/README.md', 'line_start': 1, 'line_end': 40}
[TOOL END] content='{"data": {"char_limit": 24000, "content": "# Data Onboarding Agent\\n\\n**Agent Type**: Orchestrator (Human-in-the-loop)\\n**Runs**: Per data source (repeatable)\\n**Prerequisites**: Environment setup comple

## 7. Inspect the final outcome

A successful M5.2 run should show `SUCCESS`, completed step IDs `P1`, `P2`, `P3`, and `P4`, three validated `InvestigationReport` objects, an `ArchitectureFinding`, and a passing deterministic `ValidationResult`.

A failed P4 should preserve completed P1–P3 reports and report P4 as failed. Treat any validation failure as a real finding; do not manually manufacture a success state.


In [8]:
final_state = state_snapshot.get("final") or {}
plan = state_snapshot.get("plan")
assignments = state_snapshot.get("role_assignments") or []
reports = final_state.get("reports") or []
# Backward-compatible fallback for an older/single-step graph state.
if not reports and final_state.get("report") is not None:
    reports = [final_state["report"]]

print("Final status:", final_state.get("status"))
print("Failure category:", final_state.get("failure"))
print("Error:", final_state.get("error"))
print("Completed step IDs:", final_state.get("completed_step_ids"))
print("Per-step statuses:")
pprint.pp(final_state.get("step_statuses") or {}, sort_dicts=False)

if plan is not None:
    print("\nPlan steps:")
    for step in plan.steps:
        print(f"  {step.id}: {step.goal}")

print("\nRole assignments:")
for assignment in assignments:
    print(f"  {assignment.step_id}: {assignment.role}")

print(f"\nInvestigation reports returned: {len(reports)}")
for index, report in enumerate(reports, start=1):
    print(f"\nInvestigation report {index}:")
    print("  ID:", report.investigation_id)
    print("  Status:", report.status)
    print("  Investigated sources:", report.investigated_sources)
    print("  Findings:")
    for finding in report.findings:
        print(f"    - [{finding.support}] {finding.statement}")
        for ref in finding.evidence_refs:
            print(f"        evidence: {ref.path}:{ref.line_start}-{ref.line_end}")
    print("  Open questions:", report.open_questions)
    print("  Uncertainties:", report.uncertainties)

if not reports:
    print("No validated investigation reports were returned.")

synthesis = final_state.get("architecture_finding")
validation = final_state.get("validation_result")

print("\nArchitecture synthesis:")
if synthesis is None:
    print("No validated P4 synthesis returned.")
else:
    print("  Summary:", synthesis.summary)
    print("  Source investigation IDs:", synthesis.source_investigation_ids)
    print("  Conclusions:")
    for conclusion in synthesis.conclusions:
        print(f"    - [{conclusion.classification}] {conclusion.statement}")
        for ref in conclusion.evidence_refs:
            print(f"        evidence: {ref.path}:{ref.line_start}-{ref.line_end}")

print("\nDeterministic validation:")
if validation is None:
    print("No validation result returned.")
else:
    print("  Status:", validation.status)
    for check in validation.checks:
        print(f"  - {'PASS' if check.ok else 'FAIL'} {check.rule}: {check.detail}")


Final status: SUCCESS
Failure category: None
Error: 
Completed step IDs: ['P1', 'P2', 'P3', 'P4']
Per-step statuses:
{'P1': 'completed', 'P2': 'completed', 'P3': 'completed', 'P4': 'completed'}

Plan steps:
  P1: Identify the role and functionality of agents in the system.
  P2: Explore the types of tools utilized by the agents and their purpose.
  P3: Examine how MCP servers facilitate communication and data exchange between agents and tools.
  P4: Synthesize the findings to illustrate the overall interaction dynamics among agents, tools, and MCP servers.

Role assignments:
  P1: INVESTIGATOR
  P2: INVESTIGATOR
  P3: INVESTIGATOR
  P4: ANALYST

Investigation reports returned: 3

Investigation report 1:
  ID: b79e5f78-e7c5-4a8c-8576-ff44ca1c0d47
  Status: complete
  Investigated sources: ['runbooks/data-onboarding-agent/README.md', 'SKILLS.md', 'README.md', 'shared/templates/agent_output_schema.py']
  Findings:
    - [SUPPORTED] The Data Onboarding Agent orchestrates end-to-end data pi

## 8. Inspect the safety middleware records

These are the actual middleware instances used by the live Investigator. The output is intended to help inspect budget decisions, observation bounding, and execution-guard counters.

Record shapes are owned by the project's middleware implementation; this cell prints them as-is rather than assuming every middleware version has the same fields.

In [9]:
print("Request-budget records:")
pprint.pp(investigator.budget.records, sort_dicts=False)

print("\nObservation-bounding records:")
pprint.pp(investigator.observation.records, sort_dicts=False)

print("\nExecution guard snapshot:")
pprint.pp(investigator.execution.guard.snapshot(), sort_dicts=False)


Request-budget records:
[{'run_id': None,
  'model': None,
  'message_count': 2,
  'role_counts': {'system': 1, 'user': 1},
  'messages': [{'index': 0,
                'role': 'system',
                'content_characters': 1753,
                'content_tokens_est': 439,
                'tool_call_count': 0,
                'tool_call_arguments_characters': 0},
               {'index': 1,
                'role': 'user',
                'content_characters': 245,
                'content_tokens_est': 62,
                'tool_call_count': 0,
                'tool_call_arguments_characters': 0}],
  'largest_messages': [{'index': 0, 'role': 'system', 'characters': 1753},
                       {'index': 1, 'role': 'user', 'characters': 245}],
  'tool_call_count': 0,
  'tool_call_arguments_characters': 0,
  'tool_definitions': {'count': 2, 'characters': 1168, 'tokens_est': 292},
  'structured_output': {'name': 'response_format',
                        'characters': 2966,
                

In [10]:
print("Model calls:", investigator.execution.guard.snapshot()["model_calls"])
print("Tool calls:", investigator.execution.guard.snapshot()["tool_calls"])
print("Termination reason:", investigator.execution.guard.snapshot()["termination_reason"])

for i, record in enumerate(investigator.budget.records, start=1):
    print(f"\nRequest {i}")
    print("Message count:", record["message_count"])
    print("Tool calls:", record["tool_call_count"])
    print("Disposition:", record["disposition"])

Model calls: 13
Tool calls: 12
Termination reason: TOOL_CALL_LIMIT_REACHED

Request 1
Message count: 2
Tool calls: 0
Disposition: accepted

Request 2
Message count: 4
Tool calls: 1
Disposition: accepted

Request 3
Message count: 6
Tool calls: 2
Disposition: accepted

Request 4
Message count: 8
Tool calls: 3
Disposition: accepted

Request 5
Message count: 10
Tool calls: 4
Disposition: accepted

Request 6
Message count: 12
Tool calls: 5
Disposition: accepted

Request 7
Message count: 14
Tool calls: 6
Disposition: accepted

Request 8
Message count: 16
Tool calls: 7
Disposition: accepted

Request 9
Message count: 18
Tool calls: 8
Disposition: accepted

Request 10
Message count: 20
Tool calls: 9
Disposition: accepted

Request 11
Message count: 22
Tool calls: 10
Disposition: accepted

Request 12
Message count: 24
Tool calls: 11
Disposition: accepted

Request 13
Message count: 26
Tool calls: 12
Disposition: accepted

Request 14
Message count: 2
Tool calls: 0
Disposition: accepted

Request 15


In [11]:
from importlib.metadata import version

for package in ("langchain", "langgraph", "langchain-openai"):
    print(f"{package}: {version(package)}")

langchain: 1.4.6
langgraph: 1.2.14
langchain-openai: 1.7.2


In [12]:
print("DEFAULT_MODEL:", DEFAULT_MODEL)
print("Investigator tools:", investigator.tool_names)

DEFAULT_MODEL: gpt-4o-mini
Investigator tools: ['search_repository', 'read_file']


## 9. Optional: render a graph diagram

If the installed LangGraph version supports `get_graph().draw_mermaid()`, this cell displays the outer graph's Mermaid definition. This is a structural view of the workflow, not a trace of the specific live run.

In [13]:
from IPython.display import Image, display
from langchain_core.runnables.graph import MermaidDrawMethod
try:
    mermaid = graph.get_graph().draw_mermaid()
    print(mermaid)
    #display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("Graph Mermaid rendering is unavailable in this environment.")
    print(f"{type(exc).__name__}: {exc}")


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	query_router(query_router)
	orchestrator_select(orchestrator_select)
	investigator(investigator)
	orchestrator_receive(orchestrator_receive)
	analyst(analyst)
	deterministic_validation(deterministic_validation)
	final_response(final_response)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	analyst -. &nbsp;validate&nbsp; .-> deterministic_validation;
	analyst -.-> final_response;
	deterministic_validation -.-> final_response;
	investigator -.-> final_response;
	investigator -. &nbsp;success&nbsp; .-> orchestrator_receive;
	orchestrator_receive -.-> analyst;
	orchestrator_receive -.-> final_response;
	orchestrator_receive -. &nbsp;continue&nbsp; .-> orchestrator_select;
	orchestrator_select -.-> final_response;
	orchestrator_select -. &nbsp;continue&nbsp; .-> investigator;
	planner -.-> final_response;
	planner -. &nbsp;continue&nbsp; .-> query_router;
	query_rou

## 10. What to verify after the run

Use this checklist to evaluate the live run rather than relying only on whether the notebook completed:

- [ ] `Final status` is `SUCCESS`.
- [ ] The planner generated P1, P2, P3, and P4 with meaningful goals.
- [ ] P1–P3 are assigned to `INVESTIGATOR`; P4 is assigned to `ANALYST`.
- [ ] `P1`, `P2`, and `P3` appear in `Completed step IDs`.
- [ ] Three step-specific validated `InvestigationReport` objects are returned.
- [ ] P2 focuses on tools and P3 focuses on MCP integration/protocols.
- [ ] P4 is listed as completed only if synthesis and deterministic validation pass.
- [ ] P4 source investigation IDs match the three P1-P3 reports.
- [ ] Every P4 conclusion cites evidence present in at least one P1-P3 report and attached to an Investigator finding.
- [ ] Deterministic validation reports `passed`; remember this is provenance/structure validation, not semantic entailment.
- [ ] Findings contain evidence references that actually exist in the selected repository.
- [ ] Middleware records show the expected enforcement/instrumentation.
- [ ] Any errors or uncertainty are retained rather than silently converted to success.

A successful API call does not prove that the model's findings are correct. Manually open the cited files and verify the evidence. The notebook does not claim that the full repository test suite has run.

## Troubleshooting

- **`OPENAI_API_KEY is not configured`** — set the variable in the environment that launched Jupyter, or add it to the repository-root `.env`; then rerun the credential cell.
- **`ModuleNotFoundError`** — install `pip install -e ".[poc,llm]"` in the same Python environment used by the notebook kernel, then restart the kernel.
- **Wrong repository root** — correct `REPOSITORY_ROOT` and rerun the configuration, credential, and construction cells.
- **Authentication / permission / model error** — verify that the key is valid and that the account has access to the configured model.
- **Tool reports missing architecture context or snapshot** — the live bindings deliberately refuse to borrow artifacts from a different repository. Check the project's artifact/snapshot setup and keep the repository root correct.
- **Budget or execution-limit failure** — this may be a safety control working as designed. Inspect the error and middleware records rather than removing the limits.
- **Report contains weak or incorrect claims** — check each evidence path and line range. Live model output is not deterministic and must be validated.

When asking for help, share the full traceback and the cell that raised it, but redact API keys, tokens, and private repository content.

In [14]:
from repo_intelligence.repository_query import (
    default_subject_repository,
    default_artifacts_root,
)
from repo_intelligence.retrieval.identity import resolve_index_identity

root = default_subject_repository()
artifacts = default_artifacts_root()

print("Repository root:", root.resolve())
print("Artifacts root:", artifacts.resolve())
print("Repository exists:", root.is_dir())

identity = resolve_index_identity(root)

print("Repository name:", identity.repository_name)
print("Commit identity:", identity.commit_sha)
print("Embedding model:", identity.embedding_model)

Repository root: C:\self-study\ai\build\repository-intelligence\artifacts\main_2026-09-22_ef11dac\repository\sample-Agentic-Ai-Data-Operations
Artifacts root: C:\self-study\ai\build\repository-intelligence\artifacts
Repository exists: True
Repository name: sample-Agentic-Ai-Data-Operations
Commit identity: ef11dac8618b6344d6a5e4add6ef51cd2811d7e0
Embedding model: sentence-transformers://BAAI/bge-small-en-v1.5


In [15]:
from repo_intelligence.tools.search_repository import search_repository

result = search_repository(
    {
        "query": "README",
        "max_results": 5,
    },
    repository_root=root,
)

print(result)

ToolResult(success=True, tool='search_repository', data={'query': 'README', 'results': [{'file_path': 'shared/tests/unit/test_find_relevant_memories.py', 'path': 'shared/tests/unit/test_find_relevant_memories.py', 'line_start': 163, 'line_end': 163, 'chunk_id': '19c5a01a26107d3d', 'similarity_score': 0.648691, 'language': 'python', 'repository_name': 'sample-Agentic-Ai-Data-Operations', 'commit_sha': 'ef11dac8618b6344d6a5e4add6ef51cd2811d7e0', 'snapshot_id': 'main_2026-09-22_ef11dac', 'excerpt': 'class TestMemoryLoader:'}, {'file_path': 'shared/tests/unit/test_workload_memory.py', 'path': 'shared/tests/unit/test_workload_memory.py', 'line_start': 82, 'line_end': 82, 'chunk_id': 'c5f2deed7640eecb', 'similarity_score': 0.635316, 'language': 'python', 'repository_name': 'sample-Agentic-Ai-Data-Operations', 'commit_sha': 'ef11dac8618b6344d6a5e4add6ef51cd2811d7e0', 'snapshot_id': 'main_2026-09-22_ef11dac', 'excerpt': 'class TestReadLedger:'}, {'file_path': 'shared/metadata/semantic_reader.p

In [16]:
from repo_intelligence.repository_query import live_bindings

bindings = live_bindings(
    root,
    output_root=artifacts,
)

print("Search:", bindings["search_repository"]({
    "query": "README",
    "max_results": 5,
}))

print("Architecture context:", bindings["get_architecture_context"]({
    "section": "Architecture",
    "max_chars": 2000,
}))

print("Engineering graph:", bindings["query_engineering_graph"]({
    "start_node_id": None,
}))

Search: ToolResult(success=True, tool='search_repository', data={'query': 'README', 'results': [{'file_path': 'shared/tests/unit/test_find_relevant_memories.py', 'path': 'shared/tests/unit/test_find_relevant_memories.py', 'line_start': 163, 'line_end': 163, 'chunk_id': '19c5a01a26107d3d', 'similarity_score': 0.648691, 'language': 'python', 'repository_name': 'sample-Agentic-Ai-Data-Operations', 'commit_sha': 'ef11dac8618b6344d6a5e4add6ef51cd2811d7e0', 'snapshot_id': 'main_2026-09-22_ef11dac', 'excerpt': 'class TestMemoryLoader:'}, {'file_path': 'shared/tests/unit/test_workload_memory.py', 'path': 'shared/tests/unit/test_workload_memory.py', 'line_start': 82, 'line_end': 82, 'chunk_id': 'c5f2deed7640eecb', 'similarity_score': 0.635316, 'language': 'python', 'repository_name': 'sample-Agentic-Ai-Data-Operations', 'commit_sha': 'ef11dac8618b6344d6a5e4add6ef51cd2811d7e0', 'snapshot_id': 'main_2026-09-22_ef11dac', 'excerpt': 'class TestReadLedger:'}, {'file_path': 'shared/metadata/semantic_

In [17]:
from repo_intelligence.multiagent.contracts import ArchitectureFinding

print(ArchitectureFinding.__module__)
print("source_investigation_ids" in ArchitectureFinding.model_fields)
print(list(ArchitectureFinding.model_fields.keys()))

repo_intelligence.multiagent.contracts
True
['investigation_id', 'source_investigation_ids', 'question', 'repository_scope', 'commit_scope', 'conclusions', 'evidence_refs', 'open_questions', 'summary', 'status']


In [18]:
from repo_intelligence.multiagent.contracts import (
    ArchitectureFinding,
    InvestigationReport,
)

print(
    "P4 provenance field:",
    "source_investigation_ids" in ArchitectureFinding.model_fields,
)
print(
    "Field on wrong model:",
    "source_investigation_ids" in InvestigationReport.model_fields,
)

P4 provenance field: True
Field on wrong model: False
